# Silver → Gold: dim_product

**Propósito:** Crear la dimensión de productos/líneas extrayéndola de `dbo.salestrack_sales_final` en Silver.
No existe como tabla propia en Bronze — se deriva de las columnas `line`, `agrup1`, `agrup2`.

**Lógica:**
- Clave compuesta: `agrup1` + `agrup2` + `line` (las tres columnas juntas identifican unívocamente un producto)
- Se genera un `product_id` surrogate con `row_number()`
- Se normaliza capitalización de `agrup1` y `agrup2`

**Columnas resultantes:**
- `product_id` (int) → clave primaria surrogate
- `line` → línea de producto
- `agrup1` → agrupación nivel 1
- `agrup2` → agrupación nivel 2

**Idempotencia:** `overwrite` + `overwriteSchema=true`.

In [ ]:
%run ./config

In [ ]:
SILVER_TABLE = f"{DEFAULT_SCHEMA}.salestrack_sales_final"
GOLD_TABLE   = f"{DEFAULT_SCHEMA}.dim_product"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

df_silver = spark.read.table(f"`{SILVER_LAKEHOUSE}`.{SILVER_TABLE}")

print(f"Filas fuente (salestrack): {df_silver.count()}")

In [ ]:
# ─── TRANSFORMACIONES ─────────────────────────────────────────────────────────
w = Window.orderBy("agrup1", "agrup2", "line")

df_gold = (
    df_silver
    .filter(
        F.col("line").isNotNull() &
        F.col("agrup1").isNotNull() &
        F.col("agrup2").isNotNull()
    )
    .select(
        F.trim(F.initcap(F.col("line"))).alias("line"),
        F.trim(F.initcap(F.col("agrup1"))).alias("agrup1"),
        F.trim(F.initcap(F.col("agrup2"))).alias("agrup2")
    )
    .distinct()
    .withColumn("product_id", F.row_number().over(w).cast("integer"))
    .withColumn("_gold_load_ts", F.lit(datetime.utcnow().isoformat()).cast("timestamp"))
    .select("product_id", "line", "agrup1", "agrup2", "_gold_load_ts")
)

In [ ]:
# ─── VALIDACIÓN ───────────────────────────────────────────────────────────────
row_count = df_gold.count()
null_ids  = df_gold.filter(F.col("product_id").isNull()).count()
dup_ids   = df_gold.groupBy("product_id").count().filter(F.col("count") > 1).count()

print(f"Productos únicos en Gold  : {row_count}")
print(f"product_id nulos          : {null_ids}")
print(f"product_id duplicados     : {dup_ids}")

assert null_ids == 0, "ERROR: hay product_id nulos"
assert dup_ids  == 0, "ERROR: hay product_id duplicados"

df_gold.show(20, truncate=False)

In [ ]:
# ─── ESCRITURA IDEMPOTENTE EN GOLD ────────────────────────────────────────────
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{GOLD_LAKEHOUSE}`.{GOLD_TABLE}")
)

print(f"Tabla {GOLD_TABLE} escrita en {GOLD_LAKEHOUSE} con {row_count} filas.")